<a href="https://colab.research.google.com/github/sahanameka7-rgb/GenAIDeveloper/blob/main/CrewAI_MistraiAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers accelerate torch


In [2]:
!pip uninstall -y Pillow
!pip install -U Pillow

Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 50.1 MB/s eta 0:00:00


In [1]:
from transformers import pipeline


In [2]:
# -----------------------------
# Load Hugging Face Model
# -----------------------------
# You can replace this with any supported model
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

text_generator = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name
)



config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [3]:
# -----------------------------
# Agent Class
# -----------------------------
class Agent:
    def __init__(self, role, goal, backstory):
        self.role = role
        self.goal = goal
        self.backstory = backstory

    def run(self, task_description):
        prompt = f"""
You are a {self.role}.

Goal:
{self.goal}

Background:
{self.backstory}

Task:
{task_description}

Response:
"""

        result = text_generator(
            prompt,
            max_length=150,
            num_return_sequences=1,
            truncation=True
        )

        return result[0]['generated_text']


In [4]:
# -----------------------------
# Task Class
# -----------------------------
class Task:
    def __init__(self, description, agent, expected_output=None):
        self.description = description
        self.agent = agent
        self.expected_output = expected_output

# -----------------------------
# Crew Class
# -----------------------------
class Crew:
    def __init__(self, agents, tasks, verbose=True):
        self.agents = agents
        self.tasks = tasks
        self.verbose = verbose

    def kickoff(self):
        results = []

        for task in self.tasks:
            if self.verbose:
                print(f"\nRunning Task: {task.description}")
                print(f"Agent: {task.agent.role}")

            output = task.agent.run(task.description)
            results.append(output)

            if self.verbose:
                print("Output:")
                print(output)
                print("-" * 60)

        return results




In [5]:
# -----------------------------
# Create Agents
# -----------------------------
assistant = Agent(
    role="Python Assistant",
    goal="Print 'Hello, world!' in Python.",
    backstory="You help users write and run basic Python code."
)

philosopher = Agent(
    role="Philosopher",
    goal="Share the meaning of life from literature.",
    backstory="You are inspired by 'The Hitchhiker's Guide to the Galaxy'."
)




In [6]:
# -----------------------------
# Create Tasks
# -----------------------------
task1 = Task(
    description="Print 'Hello, world!' in Python.",
    agent=assistant,
    expected_output="Hello, world!"
)

task2 = Task(
    description="Tell me the meaning of life according to popular culture.",
    agent=philosopher,
    expected_output="42"
)



In [7]:

# -----------------------------
# Create Crew
# -----------------------------
crew = Crew(
    agents=[assistant, philosopher],
    tasks=[task1, task2],
    verbose=True
)



In [8]:
# -----------------------------
# Run Crew
# -----------------------------
result = crew.kickoff()

print("\nFinal Results:")
for idx, res in enumerate(result, 1):
    print(f"\nTask {idx} Result:")
    print(res)


[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



Running Task: Print 'Hello, world!' in Python.
Agent: Python Assistant


[transformers] Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output:

You are a Python Assistant.
 
Goal:
Print 'Hello, world!' in Python.
 
Background:
You help users write and run basic Python code.
 
Task:
Print 'Hello, world!' in Python.
 
Response:
Here's the Python code to print 'Hello, world!':

```python
print('Hello, world!')
```

You can also use the following format string:

```python
print('Hello, world!')
```

Or, if you prefer, you can use parentheses and the print function like this:

```python
print('Hello, world!')
```

But whichever way you choose, the result will be the same: 'Hello, world!' will be printed to the console.
------------------------------------------------------------

Running Task: Tell me the meaning of life according to popular culture.
Agent: Philosopher
Output:

You are a Philosopher.
 
Goal:
Share the meaning of life from literature.
 
Background:
You are inspired by 'The Hitchhiker's Guide to the Galaxy'.
 
Task:
Tell me the meaning of life according to popular culture.
 
Response:
According to Douglas Ad